In [2]:
import numpy as np
import pandas as pd
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer as BagOfWords
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
pd.options.display.max_columns = 100


# NLP

If our text data are unlabelled (as is often the case in NLP), we can use KNN to identify documents that are similar to a given document.  Let's test this by creating simple documents that are similar and different.




In [1]:
%%capture
! python -m textblob.download_corpora


In [3]:
sentences = []
sentences += [ 'alpaca ' * 5 + "zebra" ]
sentences += [ 'bird ' * 5 + "zebra" ]
sentences += [ 'cat ' * 5 + "zebra" ]
sentences += [ 'dog ' * 5 + "zebra" ]
sentences += [ "alpaca bird cat dog zebra" ]
sentences += [ "alpaca bird alpaca dog zebra" ]
sentences

['alpaca alpaca alpaca alpaca alpaca zebra',
 'bird bird bird bird bird zebra',
 'cat cat cat cat cat zebra',
 'dog dog dog dog dog zebra',
 'alpaca bird cat dog zebra',
 'alpaca bird alpaca dog zebra']

# Data Cleaning


In [ ]:
# we can create data that needs cleaning

## Bag of Words Using CountVectorizer

In [4]:
# Perform the count transformation
BoW =  BagOfWords(stop_words='english')
vec = BoW.fit_transform(sentences)
vec.toarray()


array([[5, 0, 0, 0, 1],
       [0, 5, 0, 0, 1],
       [0, 0, 5, 0, 1],
       [0, 0, 0, 5, 1],
       [1, 1, 1, 1, 1],
       [2, 1, 0, 1, 1]])

In [5]:
BoW.get_feature_names_out()


array(['alpaca', 'bird', 'cat', 'dog', 'zebra'], dtype=object)

In [6]:
pd.DataFrame( vec.toarray(), columns = BoW.get_feature_names_out() )


,alpaca,bird,cat,dog,zebra
0,5,0,0,0,1
1,0,5,0,0,1
2,0,0,5,0,1
3,0,0,0,5,1
4,1,1,1,1,1
5,2,1,0,1,1


In [7]:
vec


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (6, 5)>

## TF-IDF

In [8]:
# Perform the TF-IDF transformation
tf_idf_vec = TfidfTransformer()
tf_idf_jen = tf_idf_vec.fit_transform(vec)
tf_idf_jen.toarray()


array([[0.99187771, 0.        , 0.        , 0.        , 0.12719514],
       [0.        , 0.99187771, 0.        , 0.        , 0.12719514],
       [0.        , 0.        , 0.99419024, 0.        , 0.10763724],
       [0.        , 0.        , 0.        , 0.99187771, 0.12719514],
       [0.45576869, 0.45576869, 0.53983842, 0.45576869, 0.29223139],
       [0.78988373, 0.39494186, 0.        , 0.39494186, 0.25323023]])

In [9]:
# Print out results in a dataframe
tf_df = pd.DataFrame(tf_idf_jen.toarray(), columns = BoW.get_feature_names_out())
tf_df


,alpaca,bird,cat,dog,zebra
0,0.991878,0.000000,0.000000,0.000000,0.127195
1,0.000000,0.991878,0.000000,0.000000,0.127195
2,0.000000,0.000000,0.994190,0.000000,0.107637
3,0.000000,0.000000,0.000000,0.991878,0.127195
4,0.455769,0.455769,0.539838,0.455769,0.292231
5,0.789884,0.394942,0.000000,0.394942,0.253230


## K Nearest Neighbors

In [10]:
# Fit nearest neighbors
nn = NearestNeighbors().fit(tf_idf_jen)


In [11]:
# Get nearest neighbors distances
sent0 = np.array([tf_df.iloc[0]])
distances, indices = nn.kneighbors(sent0, n_neighbors = len(tf_df))


In [12]:
distances


array([[0.        , 0.60716107, 1.01070548, 1.40272691, 1.40272691,
        1.40449925]])

In [13]:
indices


array([[0, 5, 4, 1, 3, 2]])

In [14]:
for a,b in zip(distances[0], np.array(sentences)[indices][0]):
  print(f"{a:.4f}: {b}")


0.0000: alpaca alpaca alpaca alpaca alpaca zebra
0.6072: alpaca bird alpaca dog zebra
1.0107: alpaca bird cat dog zebra
1.4027: bird bird bird bird bird zebra
1.4027: dog dog dog dog dog zebra
1.4045: cat cat cat cat cat zebra


## Euclidean distance


### In math

Pythagorean equation - in a plane - 2 dimensions
- $ \sqrt{(a_2-a_1)^2 + (b_2 - b_1)^2} $

Pythagorean equation - in space - 3 dimensions
- $ \sqrt{(a_2-a_1)^2 + (b_2 - b_1)^2 + (c_2 - c_1)^2} $

Pythagorean equation - other dimensions
- 1 dimension
  - $ \sqrt{(a_2-a_1)^2} $

- 4 dimensions
  -  $ \sqrt{(a_2-a_1)^2 + (b_2 - b_1)^2 + (c_2 - c_1)^2 + (d_2 - d_1)^2}   $

- N- dimensions
  -  $ \sqrt{(a_2-a_1)^2 + (b_2 - b_1)^2 + (c_2 - c_1)^2 + (d_2 - d_1)^2 + \ ...\  + (z_2 - z_1)^2}   $



### In Python - numpy and pandas


In [15]:
df = pd.DataFrame()

v1 = np.array([ 1, 2, 3, 4, 5, 6 ])
v2 = np.array([ 2, 4, 6, 8, 10, 12 ])

df["v2"] = v2
df["v1"] = v1
df.transpose()


,0,1,2,3,4,5
v2,2,4,6,8,10,12
v1,1,2,3,4,5,6


In [16]:
delta_v = v2 - v1
df["delta_v"] = delta_v
df.transpose()


,0,1,2,3,4,5
v2,2,4,6,8,10,12
v1,1,2,3,4,5,6
delta_v,1,2,3,4,5,6


In [17]:
delta_v2 = delta_v ** 2
df["delta_v^2"] = delta_v2
df.transpose()


,0,1,2,3,4,5
v2,2,4,6,8,10,12
v1,1,2,3,4,5,6
delta_v,1,2,3,4,5,6
delta_v^2,1,4,9,16,25,36


In [18]:
df["delta_v^2"].sum()


np.int64(91)

In [19]:
df["delta_v^2"].sum() ** 0.5


np.float64(9.539392014169456)

All in one

In [20]:
((v2 - v1)**2).sum()**0.5

np.float64(9.539392014169456)